# 5.7 — XGBoost Regression
## Predicting Crop Yield Across Tamil Nadu Farms

---

### The Story

Ravi is a data scientist at the Tamil Nadu Department of Agriculture. His team has data on hundreds of farming plots across the state — rainfall, fertilizer used, soil quality score, temperature, and farm size.

Their goal: **predict crop yield (kg per hectare)** so the government can:
- Plan procurement targets district-by-district
- Identify underperforming plots before harvest
- Recommend interventions (more fertilizer, irrigation) where needed

They tried plain Gradient Boosting — it was slow and started overfitting on their training data. A colleague suggests XGBoost. Ravi's team wants to understand *why* XGBoost is better before they switch.

---

### The Analogy: A Committee of Error-Correctors

Imagine Ravi's team has a junior analyst, Arjun, who estimates crop yield for each plot. His first guess is rough — maybe off by ₹8,000 worth of yield per hectare.

Then a second analyst, Meena, **doesn't look at the original target**. She only looks at Arjun's *mistakes* — the gap between his prediction and the real yield. She tries to predict that gap.

Then a third analyst corrects Meena's remaining mistakes. And so on.

Each analyst is a **weak learner** — not very powerful alone. But combined, they cover each other's blind spots.

**That is Gradient Boosting.**

**XGBoost** is the same idea, but the analysts are smarter:
- They use **second-order information** (not just "I'm wrong by X", but "how quickly is my error changing?") → better split decisions
- They have **built-in discipline** (L1 and L2 regularisation) → they don't overfit
- They work **faster** (parallel computation, histogram approximation)

---

### Theory: How Gradient Boosting Works (Step by Step)

Given training data with true yields **y** and predictions **ŷ**:

1. Start with a simple prediction (mean yield)
2. Compute **residuals** = y − ŷ (the errors)
3. Train Tree 2 to predict those residuals
4. Update prediction: **ŷ_new = ŷ_old + learning_rate × Tree2_prediction**
5. Compute new residuals. Train Tree 3 on those. Repeat.

Final prediction = sum of all trees (weighted by learning rate):
$$\hat{y} = T_1 + \eta \cdot T_2 + \eta \cdot T_3 + \ldots$$

Where η (eta) = learning rate.

### Where XGBoost Goes Further

| Feature | Plain GB | XGBoost |
|---|---|---|
| Optimisation | 1st derivative (gradient) only | 1st + 2nd derivative (gradient + Hessian) |
| Overfitting control | Only tree depth + learning rate | Built-in L1 (reg_alpha) + L2 (reg_lambda) |
| Missing values | Needs preprocessing | Learns the best default direction automatically |
| Speed | Sequential, slow on large data | Parallel splits, histogram approximation, cache-aware |

**Why does the 2nd derivative (Hessian) matter?**
- The gradient tells you *which direction* the error is growing
- The Hessian tells you *how fast* that direction is curving
- With both, XGBoost makes smarter split decisions — like the difference between knowing you're going uphill vs knowing the exact slope gradient at every step

---
## Step 1: Install and Import Libraries

In [ ]:
# Install xgboost if not already available
# !pip install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor  # plain GB — for comparison

import xgboost as xgb  # the star of this notebook

import warnings
warnings.filterwarnings('ignore')

print("XGBoost version:", xgb.__version__)
print("All libraries loaded.")

---
## Step 2: Create the Dataset

We simulate a realistic Tamil Nadu farming dataset. Each row = one farm plot.

In [ ]:
np.random.seed(42)  # so results are reproducible every run
n = 500             # 500 farm plots across Tamil Nadu

# Features that affect crop yield
rainfall_mm        = np.random.uniform(400, 1200, n)   # annual rainfall in mm
fertilizer_kg_ha   = np.random.uniform(50, 300, n)     # fertilizer used per hectare
soil_quality_score = np.random.uniform(1, 10, n)       # soil health score (1=poor, 10=excellent)
temperature_c      = np.random.uniform(22, 38, n)      # average growing season temperature
farm_size_ha       = np.random.uniform(0.5, 10, n)     # farm size in hectares

# True yield formula (non-linear relationships — real world is messy)
# WHY non-linear: too much fertilizer hurts (runoff), extreme heat hurts yield
yield_kg_ha = (
    2.5 * rainfall_mm
    + 3.0 * fertilizer_kg_ha
    - 0.005 * fertilizer_kg_ha**2        # fertilizer has diminishing returns
    + 80  * soil_quality_score
    - 15  * (temperature_c - 28)**2      # yield peaks at 28°C, drops away from it
    + 10  * farm_size_ha
    + np.random.normal(0, 150, n)        # real-world noise (unpredictable factors)
)

df = pd.DataFrame({
    'rainfall_mm'       : rainfall_mm,
    'fertilizer_kg_ha'  : fertilizer_kg_ha,
    'soil_quality_score': soil_quality_score,
    'temperature_c'     : temperature_c,
    'farm_size_ha'      : farm_size_ha,
    'yield_kg_ha'       : yield_kg_ha
})

print("Dataset shape:", df.shape)
df.describe().round(2)

---
## Step 3: Quick EDA — Understand the Data Before Modelling

Ravi's rule: never train a model on data you haven't looked at.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features = ['rainfall_mm', 'fertilizer_kg_ha', 'soil_quality_score',
            'temperature_c', 'farm_size_ha']

for i, feat in enumerate(features):
    ax = axes[i // 3][i % 3]
    ax.scatter(df[feat], df['yield_kg_ha'], alpha=0.3, color='steelblue', s=15)
    ax.set_xlabel(feat)
    ax.set_ylabel('Yield (kg/ha)')
    ax.set_title(f'{feat} vs Yield')

# Hide the empty 6th subplot
axes[1][2].set_visible(False)

plt.suptitle('Feature vs Yield Scatter Plots — Tamil Nadu Farm Data', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr().round(2), annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

---
## Step 4: Train-Test Split + Scaling

**Why scale before XGBoost?**

Technically, tree-based models (including XGBoost) are scale-invariant — splits use thresholds, not distances. So scaling is NOT mandatory for XGBoost.

But we scale here because:
1. Regularisation penalties (L1, L2) work more fairly when features are on the same scale
2. We'll compare with plain GB side-by-side — good practice to keep preprocessing consistent

In [ ]:
X = df.drop('yield_kg_ha', axis=1)  # all features
y = df['yield_kg_ha']               # target

# 80% train, 20% test — same split for both models so comparison is fair
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# StandardScaler: learns mean and std from TRAINING data only
# WHY only training: if we include test data, we leak future information into scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform on train
X_test_scaled  = scaler.transform(X_test)       # only transform on test (use train's mean/std)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

---
## Step 5: Plain Gradient Boosting — Baseline

Before switching to XGBoost, Ravi's team benchmarks plain Gradient Boosting.

**What happens inside `GradientBoostingRegressor.fit()`?**
1. Starts with a leaf = mean of y (the simplest possible prediction)
2. Computes residuals (y − ŷ)
3. Fits a shallow decision tree on the residuals
4. Updates predictions: ŷ += learning_rate × tree_prediction
5. Repeats for `n_estimators` trees
6. Each tree is small (controlled by `max_depth`) to stay a weak learner

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=200,    # number of sequential trees to build
    learning_rate=0.1,   # how much each tree contributes — smaller = more conservative
    max_depth=4,         # how deep each individual tree can grow — controls complexity
    random_state=42      # for reproducibility
)

gb_model.fit(X_train_scaled, y_train)

gb_preds = gb_model.predict(X_test_scaled)

gb_rmse = np.sqrt(mean_squared_error(y_test, gb_preds))
gb_mae  = mean_absolute_error(y_test, gb_preds)
gb_r2   = r2_score(y_test, gb_preds)

print("=== Plain Gradient Boosting ===")
print(f"RMSE : {gb_rmse:.2f} kg/ha")
print(f"MAE  : {gb_mae:.2f} kg/ha")
print(f"R²   : {gb_r2:.4f}")

---
## Step 6: XGBoost — The Upgrade

**What happens inside `XGBRegressor.fit()`?**

Same boosting loop as plain GB, but at each split decision:
1. Computes **gradient** (1st derivative of loss) — direction of error
2. Computes **Hessian** (2nd derivative of loss) — curvature of error
3. Uses both to calculate the **exact gain** from any potential split
4. Also applies **regularisation penalty** to the gain formula — splits that add complexity must justify themselves
5. Uses **histogram approximation** — instead of checking every possible split threshold, bins continuous features → much faster

**Key hyperparameters:**
- `n_estimators`: number of trees (more = better, but slower; use early_stopping)
- `learning_rate`: each tree's contribution weight (smaller = safer, needs more trees)
- `max_depth`: tree depth (3-6 is usually good)
- `reg_alpha`: L1 penalty (drives coefficients towards exact zero — feature selection effect)
- `reg_lambda`: L2 penalty (shrinks coefficients — smoothing effect; default=1)
- `subsample`: fraction of training rows used per tree (like RF's bootstrap — reduces overfitting)
- `colsample_bytree`: fraction of features used per tree (like RF's feature subsampling)

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=200,        # 200 sequential trees
    learning_rate=0.1,       # same as plain GB for fair comparison
    max_depth=4,             # same as plain GB for fair comparison
    reg_alpha=0.1,           # L1 regularisation — slight feature selection effect
    reg_lambda=1.0,          # L2 regularisation — default; smooths weights
    subsample=0.8,           # each tree sees 80% of training rows — reduces overfitting
    colsample_bytree=0.8,    # each tree sees 80% of features — like Random Forest's trick
    random_state=42,
    verbosity=0              # suppress training logs
)

xgb_model.fit(X_train_scaled, y_train)

xgb_preds = xgb_model.predict(X_test_scaled)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_mae  = mean_absolute_error(y_test, xgb_preds)
xgb_r2   = r2_score(y_test, xgb_preds)

print("=== XGBoost ===")
print(f"RMSE : {xgb_rmse:.2f} kg/ha")
print(f"MAE  : {xgb_mae:.2f} kg/ha")
print(f"R²   : {xgb_r2:.4f}")

---
## Step 7: Side-by-Side Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Plain Gradient Boosting', 'XGBoost'],
    'RMSE (kg/ha)': [round(gb_rmse, 2), round(xgb_rmse, 2)],
    'MAE (kg/ha)':  [round(gb_mae, 2),  round(xgb_mae, 2)],
    'R²':           [round(gb_r2, 4),   round(xgb_r2, 4)]
})

print(comparison.to_string(index=False))

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, name, color in zip(
    axes,
    [gb_preds, xgb_preds],
    ['Plain Gradient Boosting', 'XGBoost'],
    ['coral', 'steelblue']
):
    ax.scatter(y_test, preds, alpha=0.4, color=color, s=20)
    # Perfect prediction line: if model was 100% accurate, all points would fall on this line
    min_val, max_val = y_test.min(), y_test.max()
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Yield (kg/ha)')
    ax.set_ylabel('Predicted Yield (kg/ha)')
    ax.set_title(f'{name}\nR² = {r2_score(y_test, preds):.4f}')
    ax.legend()

plt.suptitle('Actual vs Predicted Yield — Plain GB vs XGBoost', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 8: Feature Importance — What Drives Yield?

**What is feature importance in XGBoost?**

XGBoost tracks, for every split across all trees:
- How many times each feature was used to split (`weight`)
- How much gain (reduction in loss) each feature contributed (`gain`) ← most meaningful
- How many samples each split covered (`cover`)

We use **gain** — it tells us which features actually *improved* predictions the most, not just which were used most often.

In [ ]:
# Extract feature importance by 'gain' — most meaningful measure
importance = xgb_model.get_booster().get_score(importance_type='gain')

feat_imp_df = pd.DataFrame(
    list(importance.items()),
    columns=['Feature', 'Gain']
).sort_values('Gain', ascending=True)  # ascending for horizontal bar chart

plt.figure(figsize=(8, 5))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Gain'], color='steelblue')
plt.xlabel('Average Gain per Split')
plt.title('XGBoost Feature Importance (by Gain)\nWhat actually drives crop yield?')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("Higher gain = this feature contributed more to reducing prediction error")
print("Ravi can now tell the Agriculture Department WHICH factors matter most for yield")

---
## Step 9: Cross-Validation — Is Our Score Reliable?

A single train-test split can get lucky or unlucky depending on which 20% of data ends up in the test set.

Cross-validation runs 5 different splits and averages the results — gives us a more honest picture of generalisation.

In [ ]:
# Scale all data for CV — WHY: CV handles the split internally, so we scale full X here
X_scaled_full = scaler.fit_transform(X)

# 5-fold CV for XGBoost — negative MSE because sklearn maximises scores (MSE should be minimised)
cv_scores = cross_val_score(
    xgb_model, X_scaled_full, y,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

cv_rmse_scores = -cv_scores  # flip sign back to positive

print("XGBoost 5-Fold Cross-Validation RMSE:")
for i, score in enumerate(cv_rmse_scores, 1):
    print(f"  Fold {i}: {score:.2f} kg/ha")
print(f"\nMean RMSE : {cv_rmse_scores.mean():.2f} kg/ha")
print(f"Std  RMSE : {cv_rmse_scores.std():.2f} kg/ha")
print("\nLow std = model performs consistently across different data splits → trustworthy")

---
## Step 10: Learning Rate Effect — Why Smaller is Safer

A common confusion: **small learning rate ≠ underfitting**.

Small learning rate means each tree contributes less. So you need MORE trees to reach the same accuracy. But each step is more careful — leads to better generalisation.

Think of it like walking downhill in the dark:
- Large steps (high LR): reach the bottom faster, but risk overshooting into a ditch
- Small steps (low LR): slower, but you feel the ground under you and avoid falling

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5]
train_rmses = []
test_rmses  = []

for lr in learning_rates:
    model = xgb.XGBRegressor(
        n_estimators=200,
        learning_rate=lr,
        max_depth=4,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train_scaled, y_train)

    train_preds = model.predict(X_train_scaled)
    test_preds  = model.predict(X_test_scaled)

    train_rmses.append(np.sqrt(mean_squared_error(y_train, train_preds)))
    test_rmses.append(np.sqrt(mean_squared_error(y_test, test_preds)))

plt.figure(figsize=(9, 5))
plt.plot(learning_rates, train_rmses, 'o-', label='Train RMSE', color='steelblue')
plt.plot(learning_rates, test_rmses,  's--', label='Test RMSE',  color='coral')
plt.xlabel('Learning Rate')
plt.ylabel('RMSE (kg/ha)')
plt.title('Learning Rate vs RMSE\n(Train vs Test — watching for overfitting)')
plt.legend()
plt.xscale('log')  # log scale because LR values span 3 orders of magnitude
plt.tight_layout()
plt.show()

print("Observation:")
print("Very high LR → test RMSE diverges from train RMSE → overfitting")
print("Very low LR with only 200 trees → underfitting (model hasn't converged yet)")
print("Sweet spot: moderate LR where test RMSE is lowest")

---
## Summary Table

| Concept | XGBoost | Plain GB |
|---|---|---|
| Base idea | Sequential trees, each corrects residuals | Same |
| Split optimisation | Gradient + Hessian (2nd order) | Gradient only (1st order) |
| Regularisation | L1 (reg_alpha) + L2 (reg_lambda) built-in | Only tree depth + LR |
| Missing values | Learned automatically | Must impute manually |
| Speed | Fast (parallel, histogram approx.) | Slower |
| Feature importance | gain / weight / cover | Impurity-based only |
| Final prediction | Sum of all trees × learning rate | Same |
| Combines by | SUM (not average) | SUM |
| Scaling needed? | Not mandatory (tree-based), but helps regularisation | Same |

---

## When to Use XGBoost

| Situation | Recommendation |
|---|---|
| Tabular data, regression or classification | XGBoost — almost always a top performer |
| Dataset has missing values | XGBoost handles them natively |
| Need feature importance | XGBoost gives gain/weight/cover options |
| Need interpretability | Use simpler model (Linear, DT) |
| Image/text/sequence data | Neural networks — XGBoost not suited |
| Very small dataset (< 100 rows) | Simpler models may generalise better |

---
## Practice Task

Ravi's team wants to experiment. Try the following:

**Task 1:** Change `reg_alpha` to 1.0 (stronger L1). Does test RMSE improve or worsen? Why?

**Task 2:** Set `subsample=1.0` and `colsample_bytree=1.0` (no subsampling). Does train RMSE drop? Does test RMSE go up? What does that tell you?

**Task 3:** Add a new feature `irrigation_score` (random uniform 1–10) that has NO real effect on yield. Does XGBoost give it zero importance? What does that tell you about its feature selection behaviour?

In [ ]:
# Task 1: Stronger L1 regularisation
# YOUR CODE HERE

# Task 2: Remove subsampling
# YOUR CODE HERE

# Task 3: Add a noise feature and check feature importance
# YOUR CODE HERE